# Run the book on Google Colab

Before the first run: upload `neurodesk_upload.zip` to the root of **My Drive**.

Runtime → Change runtime type → **GPU**. Then run every cell from top to bottom.

What is kept on Google Drive, in `My Drive/lfbook/`:

| Folder | Content |
|---|---|
| `results/` | everything the book computes (restored at the start of the next run, so finished steps are skipped) |
| `_build/` | the built book and jupyter-book's execution cache (unchanged chapters are not re-executed) |
| `records/<date>_<commit>/` | a fixed copy of each successful run: executed notebooks with their outputs + the HTML book |

## 1. Google Drive

In [ ]:
import os, shutil, subprocess
from datetime import datetime
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive')
SAVE  = DRIVE / 'lfbook'                    # results, book cache and run records
ZIP   = DRIVE / 'neurodesk_upload.zip'      # input data
BOOK  = Path('/content/new-jupyterbook')
assert ZIP.exists(), f'upload neurodesk_upload.zip to {DRIVE}'

## 2. Neurodesk tools (official Colab setup, several minutes)

In [ ]:
import json
import sys

setup_url = 'https://raw.githubusercontent.com/NeuroDesk/neurocommand/main/googlecolab_setup.sh'
subprocess.run(['curl', '-fsSL', setup_url, '-o', '/content/googlecolab_setup.sh'], check=True)
with open('/content/neurodesk-setup.log', 'w') as log:
    result = subprocess.run(
        ['bash', '/content/googlecolab_setup.sh'],
        env=dict(os.environ, NEURODESK_COLAB_PYTHON=sys.executable),
        stdout=log, stderr=subprocess.STDOUT,
    )
if result.returncode:
    print(Path('/content/neurodesk-setup.log').read_text()[-6000:])
    result.check_returncode()
os.environ.update(json.loads(Path('/content/neurodesk-colab-env.json').read_text()))
print('Neurodesk is ready. Full log: /content/neurodesk-setup.log')

## 3. Python packages not included in Colab

In [ ]:
!pip -q install "jupyter-book<2"     # v2 drops _config.yml / _toc.yml

## 4. Book code from GitHub

In [ ]:
!rm -rf {BOOK} && git clone -q https://github.com/Zane-133/new-jupyterbook.git {BOOK}
!git -C {BOOK} log -1 --format='book commit %h  %ad  %s'

## 5. WMH-SynthSeg

Same repository and commit as the original Colab runs. The model path is hard-coded to
`/app/models` and is pointed to `My Drive/wmh_model`; PyTorch ≥ 2.6 needs the same
`weights_only=False` patch as before. The model weights (790 MB) are downloaded once and kept on Drive.

In [ ]:
WMH, MODEL_DIR = '/content/wmh', DRIVE / 'wmh_model'
MODEL = MODEL_DIR / 'WMH-SynthSeg_v10_231110.pth'
INF = f'{WMH}/WMHSynthSeg/inference.py'

!rm -rf {WMH} && git clone -q https://github.com/lasopablo/freesurfer-freesurfer-dev-mri_WMHsynthseg.git {WMH} && git -C {WMH} checkout -q 2bf9a42
!sed -i "s#'/app/models'#'{MODEL_DIR}'#" {INF}
!sed -i "s/torch.load(model_file, map_location=device)/torch.load(model_file, map_location=device, weights_only=False)/" {INF}
!mkdir -p {MODEL_DIR} && (test -s {MODEL} || wget -q -c -O {MODEL} https://ftp.nmr.mgh.harvard.edu/pub/dist/lcnpublic/dist/WMH-SynthSeg/WMH-SynthSeg_v10_231110.pth)
!grep -n "model_file = \|torch.load(" {INF}
!ls -la {MODEL}

## 6. Input data: unzip from Drive to the Colab disk

In [ ]:
!rm -rf /content/neurodesk_upload && unzip -q {ZIP} -d /content
!find /content/neurodesk_upload -name '*.nii.gz' | wc -l     # expected 199

## 7. Restore earlier results and the book cache from Drive

Nothing to restore on the first run. Afterwards, finished steps are skipped and unchanged
chapters are not re-executed.

In [ ]:
def no_links(d, names):
    # symlinks (staging folders used during segmentation) are not copied; Drive cannot hold them
    return [n for n in names if os.path.islink(os.path.join(d, n))]

for name, dst in [('results', Path('/content/results')), ('_build', BOOK / '_build')]:
    src = SAVE / name
    if src.exists():
        shutil.copytree(src, dst, ignore=no_links, dirs_exist_ok=True)
        print(f'restored {src} -> {dst}')
    else:
        print(f'nothing to restore for {name}')

## 8. Run the whole book

Hours on the first run. The full jupyter-book output is in `/content/results/logs/build.log`;
only the progress lines are shown here.

In [ ]:
os.environ['BOOK_CONFIG'] = 'config_colab.yml'
!mkdir -p /content/results/logs && rm -rf {BOOK}/_build/html/reports
!cd {BOOK} && jupyter-book build . 2>&1 | tee /content/results/logs/build.log | grep --line-buffered -E "Executing|Executed|Using cached|WARNING|ERROR|build succeeded|build finished"


## 9. Save to Drive and check for errors

jupyter-book does not stop when a chapter fails, so the error reports are checked here.
Results are saved either way; a fixed record is kept only when every chapter ran without error.

In [ ]:
for name, src in [('results', Path('/content/results')), ('_build', BOOK / '_build')]:
    shutil.copytree(src, SAVE / name, ignore=no_links, dirs_exist_ok=True)
    print(f'saved {src} -> {SAVE / name}')

reports = sorted((BOOK / '_build/html/reports').glob('*.err.log'))
if reports:
    for r in reports:
        print(f'\n===== {r.name} (last lines) =====')
        print(''.join(r.read_text().splitlines(keepends=True)[-25:]))
    raise RuntimeError(f'{len(reports)} chapter(s) failed; see above')

commit = subprocess.run(['git', '-C', str(BOOK), 'rev-parse', '--short', 'HEAD'],
                        capture_output=True, text=True).stdout.strip()
record = SAVE / 'records' / f"{datetime.now():%Y%m%d-%H%M}_{commit}"
for sub in ['jupyter_execute', 'html']:                # executed notebooks, HTML book
    if (BOOK / '_build' / sub).exists():
        shutil.copytree(BOOK / '_build' / sub, record / sub, ignore=no_links)
print(f'all chapters ran; record kept in {record}')